In [1]:
import os
import json
import data
import random
import sklearn
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from sklearn.decomposition import NMF
from sklearn.utils import shuffle, check_random_state

import config
from utils import apply_symmetric_noise

random.seed(config.SEED)
np.random.seed(config.SEED)
np.random.default_rng(config.SEED)
check_random_state(config.SEED)

import warnings

warnings.filterwarnings("ignore")

In [2]:
DATA_SIZE = 3000
NOISE_PROB = 0.15
MAX_ITER = 10000
COLLECTION_NAME = "Swimmer"
DATA_FOLDER_PATH = Path(f"NMF_{COLLECTION_NAME}_{DATA_SIZE}_{str(NOISE_PROB).replace('.', '')}_denoise_info_test")
os.makedirs(DATA_FOLDER_PATH, exist_ok=True)

In [3]:
if COLLECTION_NAME == "UTK":
    array = shuffle(data.load_utk())[:DATA_SIZE]
elif COLLECTION_NAME == "Olivetti":
    array = shuffle(data.load_olivetti())[:DATA_SIZE]
elif COLLECTION_NAME == "Swimmer":
    array = shuffle(data.load_swimmer())[:DATA_SIZE]
else:
    raise ValueError

In [4]:
array_noisy = apply_symmetric_noise(array, prob=NOISE_PROB)

In [5]:
def cosine_distance(u: np.ndarray, v: np.ndarray) -> float:
    u = np.asarray(u, dtype=np.float64).ravel()
    v = np.asarray(v, dtype=np.float64).ravel()
    uu = np.dot(u, u); vv = np.dot(v, v)
    if uu <= 0.0 or vv <= 0.0:
        return 1.0
    return 1.0 - (np.dot(u, v) / np.sqrt(uu * vv))

In [6]:
def calc_deltas(X_clean: np.ndarray, X_noisy: np.ndarray, X_hat_noisy: np.ndarray) -> np.ndarray:
    M = X_clean.shape[0]
    deltas = np.empty(M, dtype=np.float64)
    for i in range(M):
        d1 = cosine_distance(X_clean[i], X_noisy[i])
        d2 = cosine_distance(X_clean[i], X_hat_noisy[i])
        deltas[i] = d1 - d2
    return deltas

In [7]:
def _row_normalize(X: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    Xn = np.divide(X, norms, out=np.zeros_like(X), where=norms > 0)
    return Xn

def pairwise_cosine_distance_matrix(XA: np.ndarray, XB: np.ndarray) -> np.ndarray:
    A = _row_normalize(XA)
    B = _row_normalize(XB)
    sims = A @ B.T
    np.clip(sims, -1.0, 1.0, out=sims)
    return 1.0 - sims

def retrieval_accuracy(clean: np.ndarray, recon: np.ndarray, topk=(1, 5)) -> dict:
    assert clean.shape == recon.shape, "clean and recon must have same shape (M, N)"
    M = clean.shape[0]
    D = pairwise_cosine_distance_matrix(clean, recon)
    
    nearest_rows_by_col = np.argsort(D, axis=0)
    results = {}
    arangeM = np.arange(M)
    for k in topk:
        hits = np.any(nearest_rows_by_col[:k, :] == arangeM[None, :], axis=0)
        results[f"top{k}"] = float(hits.mean())
    return results

In [8]:
for rank in tqdm(range(1, 200)):
    # Calculations
    model = NMF(n_components=rank, max_iter=MAX_ITER, init="random", random_state=config.SEED)
    
    W = model.fit_transform(array_noisy)
    H = model.components_
    
    array_recon = W @ H
    
    delt = calc_deltas(array, array_noisy, array_recon)
    
    acc_denoised = retrieval_accuracy(array, array_recon, topk=(1, 5, 10))
    acc_noisy = retrieval_accuracy(array, array_noisy, topk=(1, 5, 10))

    # Create result
    result = dict()
    result["rank"] = rank
    result["delta_mean"] = np.mean(delt).item()
    result["delta_var"] = np.var(delt, ddof=1).item()
    result["delta_positive_num"] = int((delt > 0).sum())
    
    for key in acc_denoised:
        result[f"denoised_{key}"] = acc_denoised[key]
        result[f"noisy_{key}"] = acc_noisy[key]

    # Save results
    file_name = f"{COLLECTION_NAME}-{DATA_SIZE}-{str(NOISE_PROB).replace('.', '_')}-{rank}.json"
    file_path = os.path.join(DATA_FOLDER_PATH, file_name)

    with open(file_path, "w") as f:
        json.dump(result, f)

100%|█████████████████████████████████████████| 199/199 [07:23<00:00,  2.23s/it]
